In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
import matplotlib.pyplot as plt


In [2]:
def seismic(df):
    s = df[
        [
            "time",
            "latitude",
            "longitude",
            "depth_km",
            "magnitude",
            "coast delta",
            "fault delta",
            "ETAS",
        ]
    ].copy()

    return s.sort_values("time").reset_index(drop=True)




DATA_DIR = Path("./Data")   # <-- *pointed* at the folder with the three CSVs

def load(split):
    df = pd.read_csv(DATA_DIR / f"earthquakeq_{split}_feature_fitted.csv", parse_dates=["time"])
    return df

train_raw = load("train")

s = seismic(train_raw)

print(s.columns.tolist())


test_raw  = load("test")
judge_raw = load("judge")

for name, df in [("train", train_raw), ("test", test_raw), ("judge", judge_raw)]:
    print(f"{name:6s}: {len(df):>7,} rows | sources: {dict(df.source.value_counts())}")

['time', 'latitude', 'longitude', 'depth_km', 'magnitude', 'coast delta', 'fault delta', 'ETAS']
train :  11,597 rows | sources: {'seismic': np.int64(11597)}
test  :   2,027 rows | sources: {'seismic': np.int64(2027)}
judge :     740 rows | sources: {'seismic': np.int64(740)}


## 7 · Your turn

You now have a complete, honest, working forecast. It is **plain on purpose.** Some directions teams
often explore (no promises — you decide what's worth it):

- **Richer seismicity features** — b-values, event-rate acceleration, moment release, multi-radius counts.
- **The other four data streams** — geomagnetic, solar, tidal, GPS. Do they add anything? Prove it, don't assume it.
- **A physics baseline** — the ETAS aftershock model is a classic; feeding its forecast in as a feature is a strong move.
- **Better calibration and model selection** — how you split time and calibrate can matter more than the model.
- **Quantum approaches** — encodings, kernels, variational classifiers, if your track calls for it.

**Two things worth remembering:**
1. Report your gains *with* calibration, and always check your held-out (judge) number against your
   own test number — if judge is far worse, you probably leaked.
2. If your fancy pipeline doesn't clearly beat this simple one, that's a real and interesting result.
   Say so, honestly, with the numbers to back it up.

Good luck. Build something you can defend.

QBits:

In [3]:
import os
import random

import numpy as np
import pandas as pd
import torch
import pennylane as qml

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    confusion_matrix,
)

from pennylane.qnn import TorchLayer


# ============================================================
# Configuration
# ============================================================

SEED = 69

# Five input features:
#   latitude
#   longitude
#   fault delta
#   coast delta
#   depth_km
#
# No PCA is used.
#
# Therefore the quantum circuit also uses five qubits.
N_QUBITS = 4
N_LAYERS = 3

MAGNITUDE_THRESHOLD = 2.0
RADIUS_KM = 100.0
HORIZON_DAYS = 30

BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 0.001

CLASSIFICATION_THRESHOLD = 0.5

# Local simulator.
SHOTS = 1024


# ============================================================
# Reproducibility
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ============================================================
# Label inspection utility
# ============================================================

def inspect_labels(name, y):
    values, counts = np.unique(y, return_counts=True)

    print(f"\n{name}")
    print("shape:", y.shape)
    print("unique values:", values)
    print("counts:", dict(zip(values, counts)))
    print("positive rate:", float(np.mean(y)))


# ============================================================
# Basic validation
# ============================================================

def validate_columns(df, name):
    """
    Validate that all columns required by the pipeline exist.
    """

    required = {
        "time",
        "latitude",
        "longitude",
        "magnitude",
        "depth_km",
        "coast delta",
        "fault delta",
    }

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{name} is missing required columns: "
            f"{sorted(missing)}. "
            f"Available columns are: {list(df.columns)}"
        )


# ============================================================
# Geographic utility
# ============================================================

def haversine_km(
    lat1,
    lon1,
    lat2,
    lon2,
):
    """
    Calculate great-circle distance in kilometers.

    lat1/lon1 may be scalars.
    lat2/lon2 may be NumPy arrays.
    """

    earth_radius_km = 6371.0088

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)

    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2.0) ** 2
    )

    return (
        2.0
        * earth_radius_km
        * np.arcsin(np.sqrt(a))
    )


# ============================================================
# Catalog cleaning
# ============================================================

def clean_catalog(df, name):
    """
    Clean a seismic catalog.

    Only the five requested feature columns are eventually
    passed to the machine-learning model.

    The time and magnitude columns are retained because they
    are required for constructing the future-event labels.
    """

    df = df.copy()

    validate_columns(df, name)

    # --------------------------------------------------------
    # Normalize timestamps
    # --------------------------------------------------------

    df["time"] = (
        pd.to_datetime(
            df["time"],
            errors="coerce",
            utc=True,
        )
        .dt.tz_localize(None)
    )

    # --------------------------------------------------------
    # Convert required numeric columns
    # --------------------------------------------------------

    numeric_columns = [
        "latitude",
        "longitude",
        "magnitude",
        "depth_km",
        "coast delta",
        "fault delta",
        "ETAS",
    ]

    for column in numeric_columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

    # --------------------------------------------------------
    # Drop rows that cannot be used
    # --------------------------------------------------------

    df = df.dropna(
        subset=[
            "time",
            "latitude",
            "longitude",
            "magnitude",
            "depth_km",
            "coast delta",
            "fault delta",
            "ETAS",
        ]
    )

    return (
        df
        .sort_values("time")
        .reset_index(drop=True)
    )


# ============================================================
# Future-event label generation
# ============================================================

def make_future_event_labels(
    reference_df,
    earthquake_df,
    radius_km=100.0,
    horizon_days=30,
    magnitude_threshold=2.0,
):
    """
    Label each reference event as:

        1 = an earthquake above magnitude_threshold occurs
            within radius_km during the future horizon.

        0 = no such earthquake occurs.

    The event at the reference timestamp itself is not included
    because the time condition is:

        event_time > reference_time

    and

        event_time <= reference_time + horizon.
    """

    reference_df = reference_df.copy()
    earthquake_df = earthquake_df.copy()

    # --------------------------------------------------------
    # Normalize timestamps
    # --------------------------------------------------------

    reference_df["time"] = (
        pd.to_datetime(
            reference_df["time"],
            errors="coerce",
            utc=True,
        )
        .dt.tz_localize(None)
    )

    earthquake_df["time"] = (
        pd.to_datetime(
            earthquake_df["time"],
            errors="coerce",
            utc=True,
        )
        .dt.tz_localize(None)
    )

    # --------------------------------------------------------
    # Clean reference catalog
    # --------------------------------------------------------

    reference_df = (
        reference_df
        .dropna(
            subset=[
                "time",
                "latitude",
                "longitude",
            ]
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Clean earthquake catalog
    # --------------------------------------------------------

    earthquake_df = (
        earthquake_df
        .dropna(
            subset=[
                "time",
                "latitude",
                "longitude",
                "magnitude",
            ]
        )
        .sort_values("time")
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Convert event information to NumPy arrays
    # --------------------------------------------------------

    event_times = earthquake_df["time"]

    event_latitudes = (
        earthquake_df["latitude"].to_numpy()
    )

    event_longitudes = (
        earthquake_df["longitude"].to_numpy()
    )

    event_magnitudes = (
        earthquake_df["magnitude"].to_numpy()
    )

    # --------------------------------------------------------
    # Allocate labels
    # --------------------------------------------------------

    labels = np.zeros(
        len(reference_df),
        dtype=np.int64,
    )

    # --------------------------------------------------------
    # Generate labels
    # --------------------------------------------------------

    for i, reference_row in reference_df.iterrows():

        start_time = reference_row["time"]

        end_time = (
            start_time
            + pd.Timedelta(days=horizon_days)
        )

        # Candidate future earthquakes satisfying:
        #
        #   1. Future relative to reference event
        #   2. Inside prediction horizon
        #   3. Above magnitude threshold

        candidate_mask = (
            (event_times > start_time)
            & (event_times <= end_time)
            & (
                event_magnitudes
                > magnitude_threshold
            )
        )

        candidate_indices = np.flatnonzero(
            candidate_mask.to_numpy()
        )

        if len(candidate_indices) == 0:
            continue

        # ----------------------------------------------------
        # Geographic distance
        # ----------------------------------------------------

        distances = haversine_km(
            reference_row["latitude"],
            reference_row["longitude"],
            event_latitudes[candidate_indices],
            event_longitudes[candidate_indices],
        )

        # Positive label if at least one qualifying event
        # occurs within the requested radius.

        labels[i] = int(
            np.any(distances <= radius_km)
        )

    return labels


# ============================================================
# Feature matrix
# ============================================================

def make_feature_matrix(df):
    """
    Build the machine-learning feature matrix.

    IMPORTANT:
    These are the ONLY five features used for training.

        1. latitude
        2. longitude
        3. fault delta
        4. coast delta
        5. depth_km

    No time features are generated.

    No magnitude feature is used.

    No PCA is performed.
    """

    feature_columns = [
        #"latitude",
        #"longitude",
        "fault delta",
        "coast delta",
        "depth_km",
        "ETAS",
    ]

    missing = (
        set(feature_columns)
        - set(df.columns)
    )

    if missing:
        raise ValueError(
            f"Missing feature columns: "
            f"{sorted(missing)}"
        )

    return df[
        feature_columns
    ].to_numpy(
        dtype=np.float32
    )


# ============================================================
# Load and prepare catalogs
# ============================================================

# These objects/functions must already exist in your
# environment:
#
#   train_raw
#   test_raw
#   judge_raw
#   seismic(...)
#

cat_train = clean_catalog(
    seismic(train_raw),
    "training catalog",
)

cat_test = clean_catalog(
    seismic(test_raw),
    "test catalog",
)

cat_judge = clean_catalog(
    seismic(judge_raw),
    "judge catalog",
)


# ============================================================
# Create binary labels
# ============================================================

y_train = make_future_event_labels(
    reference_df=cat_train,
    earthquake_df=cat_train,
    radius_km=RADIUS_KM,
    horizon_days=HORIZON_DAYS,
    magnitude_threshold=MAGNITUDE_THRESHOLD,
)

y_test = make_future_event_labels(
    reference_df=cat_test,
    earthquake_df=cat_test,
    radius_km=RADIUS_KM,
    horizon_days=HORIZON_DAYS,
    magnitude_threshold=MAGNITUDE_THRESHOLD,
)

y_judge = make_future_event_labels(
    reference_df=cat_judge,
    earthquake_df=cat_judge,
    radius_km=RADIUS_KM,
    horizon_days=HORIZON_DAYS,
    magnitude_threshold=MAGNITUDE_THRESHOLD,
)


# ============================================================
# Inspect labels
# ============================================================

inspect_labels(
    "Training labels",
    y_train,
)

inspect_labels(
    "Test labels",
    y_test,
)

inspect_labels(
    "Judge labels",
    y_judge,
)


# ============================================================
# Create the five-feature matrices
# ============================================================

x_train = make_feature_matrix(
    cat_train
)

x_test = make_feature_matrix(
    cat_test
)

x_judge = make_feature_matrix(
    cat_judge
)


# ============================================================
# Chronological ordering
# ============================================================

# Each catalog is independently sorted chronologically.
#
# No random shuffling occurs before preprocessing.

train_order = np.argsort(
    cat_train["time"].to_numpy()
)

test_order = np.argsort(
    cat_test["time"].to_numpy()
)

judge_order = np.argsort(
    cat_judge["time"].to_numpy()
)


x_train = x_train[
    train_order
]

y_train = y_train[
    train_order
]


x_test = x_test[
    test_order
]

y_test = y_test[
    test_order
]


x_judge = x_judge[
    judge_order
]

y_judge = y_judge[
    judge_order
]


# ============================================================
# Imputation
# ============================================================

# Median imputation is fitted ONLY on the training set.

imputer_x = SimpleImputer(
    strategy="median"
)

x_train_imputed = (
    imputer_x.fit_transform(
        x_train
    )
)

x_test_imputed = (
    imputer_x.transform(
        x_test
    )
)

x_judge_imputed = (
    imputer_x.transform(
        x_judge
    )
)


# ============================================================
# Standardization
# ============================================================

# StandardScaler is fitted ONLY on the training set.
#
# The five features remain five features.
#
# There is NO dimensionality reduction.

scaler_x = StandardScaler()

x_train_scaled = (
    scaler_x.fit_transform(
        x_train_imputed
    )
)

x_test_scaled = (
    scaler_x.transform(
        x_test_imputed
    )
)

x_judge_scaled = (
    scaler_x.transform(
        x_judge_imputed
    )
)


# ============================================================
# Verify feature dimensionality
# ============================================================

assert x_train_scaled.shape[1] == N_QUBITS
assert x_test_scaled.shape[1] == N_QUBITS
assert x_judge_scaled.shape[1] == N_QUBITS

print(
    "\nFeature configuration:"
)

print(
    "Features:",
    [
        "latitude",
        "longitude",
        "fault delta",
        "coast delta",
        "depth_km",
        "ETAS",
    ],
)

print(
    "Number of classical features:",
    x_train_scaled.shape[1],
)

print(
    "Number of qubits:",
    N_QUBITS,
)

print(
    "PCA:",
    "DISABLED",
)


# ============================================================
# Convert data to PyTorch tensors
# ============================================================

X_train_t = torch.tensor(
    x_train_scaled,
    dtype=torch.float32,
)

X_test_t = torch.tensor(
    x_test_scaled,
    dtype=torch.float32,
)

X_judge_t = torch.tensor(
    x_judge_scaled,
    dtype=torch.float32,
)


Y_train_t = torch.tensor(
    y_train.reshape(-1, 1),
    dtype=torch.float32,
)

Y_test_t = torch.tensor(
    y_test.reshape(-1, 1),
    dtype=torch.float32,
)

Y_judge_t = torch.tensor(
    y_judge.reshape(-1, 1),
    dtype=torch.float32,
)


# ============================================================
# Training data loader
# ============================================================

dataset_train = torch.utils.data.TensorDataset(
    X_train_t,
    Y_train_t,
)

loader_train = torch.utils.data.DataLoader(
    dataset_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
)


# ============================================================
# PennyLane simulator
# ============================================================

# Five features -> five qubits.
#
# This is a local simulator.
#
# No IBM hardware, account, token, or network connection
# is required.

dev = qml.device(
    "default.qubit",
    wires=N_QUBITS,
    shots=SHOTS,
)


# ============================================================
# Quantum circuit
# ============================================================

@qml.qnode(
    dev,
    interface="torch",
    diff_method="parameter-shift",
)
def circuit(
    inputs,
    weights,
):
    # Five standardized classical features are directly
    # angle-encoded onto five qubits.
    qml.AngleEmbedding(
        inputs,
        wires=range(N_QUBITS),
    )

    qml.BasicEntanglerLayers(
        weights,
        wires=range(N_QUBITS),
    )

    return [
        qml.expval(
            qml.PauliZ(i)
        )
        for i in range(N_QUBITS)
    ]


# ============================================================
# Quantum layer
# ============================================================

weight_shapes = {
    "weights": (
        N_LAYERS,
        N_QUBITS,
    )
}

q_layer = TorchLayer(
    circuit,
    weight_shapes,
)


# ============================================================
# Hybrid quantum-classical classifier
# ============================================================

class HybridClassifier(
    torch.nn.Module
):
    def __init__(
        self,
        quantum_layer,
    ):
        super().__init__()

        self.quantum_layer = (
            quantum_layer
        )

        self.output_layer = (
            torch.nn.Linear(
                N_QUBITS,
                1,
            )
        )

    def forward(self, x):

        quantum_output = (
            self.quantum_layer(x)
        )

        # Raw logit.
        #
        # Do NOT apply sigmoid here because
        # BCEWithLogitsLoss applies it internally.

        return self.output_layer(
            quantum_output
        )


# ============================================================
# Initialize model
# ============================================================

model = HybridClassifier(
    q_layer
)


# ============================================================
# Loss and optimizer
# ============================================================

loss_fn = (
    torch.nn.BCEWithLogitsLoss()
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
)


# ============================================================
# Training
# ============================================================

last_loss = None

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0.0

    for xb, yb in loader_train:

        optimizer.zero_grad()

        logits = model(xb)

        loss = loss_fn(
            logits,
            yb,
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item()
            * len(xb)
        )

    average_loss = (
        total_loss
        / len(dataset_train)
    )

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} "
        f"- loss: {average_loss:.4f}"
    )

    # Stop if the loss has stopped changing
    # at four decimal places.

    rounded_loss = round(
        average_loss,
        4,
    )

    if (
        last_loss is not None
        and rounded_loss == last_loss
    ):
        print(
            "Training stopped early: "
            "loss unchanged."
        )
        break

    last_loss = rounded_loss


# ============================================================
# Expected Calibration Error
# ============================================================

def expected_calibration_error(
    y_true,
    probabilities,
    n_bins=10,
):
    """
    Calculate Expected Calibration Error (ECE).

    Predictions are divided into equally spaced probability
    bins between 0 and 1.

    For each non-empty bin:

        calibration error =
            |mean predicted probability
             - empirical positive rate|

    ECE is the weighted average of these errors.
    """

    y_true = np.asarray(
        y_true
    ).reshape(-1)

    probabilities = np.asarray(
        probabilities
    ).reshape(-1)

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    ece = 0.0

    n = len(y_true)

    for i in range(n_bins):

        if i == n_bins - 1:

            mask = (
                (
                    probabilities
                    >= bin_edges[i]
                )
                & (
                    probabilities
                    <= bin_edges[i + 1]
                )
            )

        else:

            mask = (
                (
                    probabilities
                    >= bin_edges[i]
                )
                & (
                    probabilities
                    < bin_edges[i + 1]
                )
            )

        if not np.any(mask):
            continue

        bin_probabilities = (
            probabilities[mask]
        )

        bin_labels = (
            y_true[mask]
        )

        mean_probability = (
            np.mean(
                bin_probabilities
            )
        )

        empirical_accuracy = (
            np.mean(
                bin_labels
            )
        )

        ece += (
            np.sum(mask) / n
        ) * abs(
            mean_probability
            - empirical_accuracy
        )

    return float(ece)


# ============================================================
# Model evaluation
# ============================================================

def evaluate_model(
    model,
    X,
    y,
    threshold=0.5,
    baseline_positive_rate=None,
    ece_bins=10,
):
    """
    Evaluate the trained classifier.

    Returns:

        accuracy
        precision
        recall
        F1
        ROC-AUC
        PR-AUC
        log loss
        information gain in bits
        Brier score
        ECE
        confusion matrix
        probabilities
        predictions
    """

    model.eval()

    with torch.no_grad():

        logits = model(X)

        probabilities = (
            torch.sigmoid(logits)
            .reshape(-1)
            .cpu()
            .numpy()
        )

    predictions = (
        probabilities >= threshold
    ).astype(np.int64)

    y_true = (
        y.reshape(-1)
        .astype(np.int64)
    )

    # --------------------------------------------------------
    # Classification metrics
    # --------------------------------------------------------

    results = {

        "accuracy": accuracy_score(
            y_true,
            predictions,
        ),

        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0,
        ),

        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0,
        ),

        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0,
        ),

        "confusion_matrix": (
            confusion_matrix(
                y_true,
                predictions,
            )
        ),

        "probabilities": probabilities,

        "predictions": predictions,
    }

    # --------------------------------------------------------
    # ROC-AUC and PR-AUC
    # --------------------------------------------------------

    if len(
        np.unique(y_true)
    ) == 2:

        results["auc"] = (
            roc_auc_score(
                y_true,
                probabilities,
            )
        )

        results["pr_auc"] = (
            average_precision_score(
                y_true,
                probabilities,
            )
        )

    else:

        results["auc"] = float(
            "nan"
        )

        results["pr_auc"] = float(
            "nan"
        )

    # --------------------------------------------------------
    # Brier score
    # --------------------------------------------------------

    results["brier"] = (
        brier_score_loss(
            y_true,
            probabilities,
        )
    )

    # --------------------------------------------------------
    # Log loss
    # --------------------------------------------------------

    probabilities_clipped = np.clip(
        probabilities,
        1e-15,
        1.0 - 1e-15,
    )

    model_log_loss = log_loss(
        y_true,
        probabilities_clipped,
        labels=[0, 1],
    )

    # --------------------------------------------------------
    # Baseline probability
    # --------------------------------------------------------

    if baseline_positive_rate is None:

        baseline_positive_rate = (
            float(
                np.mean(y_true)
            )
        )

    baseline_positive_rate = np.clip(
        baseline_positive_rate,
        1e-15,
        1.0 - 1e-15,
    )

    baseline_probabilities = np.full(
        len(y_true),
        baseline_positive_rate,
        dtype=np.float64,
    )

    baseline_log_loss = log_loss(
        y_true,
        baseline_probabilities,
        labels=[0, 1],
    )

    # --------------------------------------------------------
    # Information gain
    # --------------------------------------------------------

    # Convert from nats to bits.

    results["ig_bits"] = (
        baseline_log_loss
        - model_log_loss
    ) / np.log(2.0)

    results["log_loss"] = (
        model_log_loss
    )

    # --------------------------------------------------------
    # Expected Calibration Error
    # --------------------------------------------------------

    results["ece"] = (
        expected_calibration_error(
            y_true=y_true,
            probabilities=probabilities,
            n_bins=ece_bins,
        )
    )

    return results


# ============================================================
# Evaluation baseline
# ============================================================

# The training-set event rate is used as the baseline
# probability for information-gain calculations.

train_positive_rate = float(
    np.mean(y_train)
)

print(
    f"\nTraining positive rate: "
    f"{train_positive_rate:.6f}"
)


# ============================================================
# Test evaluation
# ============================================================

test_results = evaluate_model(
    model=model,
    X=X_test_t,
    y=y_test,
    threshold=0.65,
    baseline_positive_rate=(
        train_positive_rate
    ),
    ece_bins=10,
)


# ============================================================
# Judge evaluation
# ============================================================

judge_results = evaluate_model(
    model=model,
    X=X_judge_t,
    y=y_judge,
    threshold=CLASSIFICATION_THRESHOLD,
    baseline_positive_rate=(
        train_positive_rate
    ),
    ece_bins=10,
)


# ============================================================
# Test results
# ============================================================

print("\nTest results")

print(
    f"AUC:         "
    f"{test_results['auc']:.4f}"
)

print(
    f"PR-AUC:      "
    f"{test_results['pr_auc']:.4f}"
)

print(
    f"IG_bits:     "
    f"{test_results['ig_bits']:.4f}"
)

print(
    f"Brier:       "
    f"{test_results['brier']:.4f}"
)

print(
    f"ECE:         "
    f"{test_results['ece']:.4f}"
)

print(
    f"Accuracy:    "
    f"{test_results['accuracy']:.4f}"
)

print(
    f"Precision:   "
    f"{test_results['precision']:.4f}"
)

print(
    f"Recall:      "
    f"{test_results['recall']:.4f}"
)

print(
    f"F1 score:    "
    f"{test_results['f1']:.4f}"
)

print(
    "Confusion matrix:"
)

print(
    test_results[
        "confusion_matrix"
    ]
)


# ============================================================
# Judge results
# ============================================================

print("\nJudge results")

print(
    f"AUC:         "
    f"{judge_results['auc']:.4f}"
)

print(
    f"PR-AUC:      "
    f"{judge_results['pr_auc']:.4f}"
)

print(
    f"IG_bits:     "
    f"{judge_results['ig_bits']:.4f}"
)

print(
    f"Brier:       "
    f"{judge_results['brier']:.4f}"
)

print(
    f"ECE:         "
    f"{judge_results['ece']:.4f}"
)

print(
    f"Accuracy:    "
    f"{judge_results['accuracy']:.4f}"
)

print(
    f"Precision:   "
    f"{judge_results['precision']:.4f}"
)

print(
    f"Recall:      "
    f"{judge_results['recall']:.4f}"
)

print(
    f"F1 score:    "
    f"{judge_results['f1']:.4f}"
)

print(
    "Confusion matrix:"
)

print(
    judge_results[
        "confusion_matrix"
    ]
)


# ============================================================
# Inspect judge predictions
# ============================================================

judge_predictions = pd.DataFrame({

    "actual":
        y_judge,

    "probability_event_within_100km":
        judge_results[
            "probabilities"
        ],

    "predicted_class":
        judge_results[
            "predictions"
        ],
})

print(
    "\nPrediction sample:"
)

print(
    judge_predictions.head(10)
)



Training labels
shape: (11582,)
unique values: [0 1]
counts: {np.int64(0): np.int64(3878), np.int64(1): np.int64(7704)}
positive rate: 0.6651700915213262

Test labels
shape: (2027,)
unique values: [0 1]
counts: {np.int64(0): np.int64(593), np.int64(1): np.int64(1434)}
positive rate: 0.7074494326591021

Judge labels
shape: (740,)
unique values: [0 1]
counts: {np.int64(0): np.int64(281), np.int64(1): np.int64(459)}
positive rate: 0.6202702702702703

Feature configuration:
Features: ['latitude', 'longitude', 'fault delta', 'coast delta', 'depth_km', 'ETAS']
Number of classical features: 4
Number of qubits: 4
PCA: DISABLED
Epoch 01/30 - loss: 0.6259
Epoch 02/30 - loss: 0.5449
Epoch 03/30 - loss: 0.5184
Epoch 04/30 - loss: 0.5088
Epoch 05/30 - loss: 0.5045
Epoch 06/30 - loss: 0.5025
Epoch 07/30 - loss: 0.5009
Epoch 08/30 - loss: 0.5000
Epoch 09/30 - loss: 0.4988
Epoch 10/30 - loss: 0.4979
Epoch 11/30 - loss: 0.4973
Epoch 12/30 - loss: 0.4969
Epoch 13/30 - loss: 0.4966
Epoch 14/30 - loss: 0